# Week 9 — Classification

This notebook goes with the Week 9 page. It uses the same dataset, the same split and the same unknown point as the class demonstration. Here you can change the code, run it again, and see what happens.

Each grey box is a **code cell**. Click a cell and press **Shift + Enter** to run it. Run the cells in order, from top to bottom.

**In Google Colab**, everything this notebook needs is already installed. A notebook opened from GitHub does not keep your changes, so choose **File → Save a copy in Drive** before you start. Colab also disconnects after a while if you leave it idle. If that happens, run the cells again from the top.

**In VS Code**, install the Python and Jupyter extensions, open this file, and choose a Python environment as the kernel. Install the libraries once, by running `python -m pip install numpy pandas scikit-learn matplotlib` in a terminal.

## 1. Set up

`pandas` is new this week. It holds results in tables, which makes it easy to compare models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

## 2. The data

`make_moons` generates two interlocking crescents of points. Fixing `random_state` means you get exactly the same points every time, and the same points as the Week 9 page.

The labels are the numbers 0 and 1. On the page they are called A and B.

In [ ]:
X, y = make_moons(n_samples=140, noise=0.23, random_state=2)
names = np.array(["A", "B"])
unknown = np.array([[-0.4, 0.15]])

print("X has shape", X.shape)
print("y has shape", y.shape)
print("First five rows of X:")
print(X[:5])
print("First five labels:", y[:5])
print("Examples in each class:", np.bincount(y))

Look at the output before you go on. Each row of `X` is one example with two features. The two classes have 70 examples each.

## 3. Split into training and test data

Before any learning happens, we hold back 30% of the examples as a test set. The model never sees them during training, so they give a fair check of how it does on new data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Training set:", X_train.shape, "Test set:", X_test.shape)
print("Training examples in each class:", np.bincount(y_train))
print("Test examples in each class:", np.bincount(y_test))

`stratify=y` keeps the same mix of the two classes in both parts. Try removing it later and see how the counts change.

## 4. Look at the data

This cell defines a plotting function, which the rest of the notebook reuses. Solid points are training examples, hollow points are test examples, and the diamond is the unknown point. When you pass it a fitted model, it also shades the region the model gives to each class.

In [ ]:
def plot_model(model=None, title=""):
    fig, ax = plt.subplots(figsize=(8, 5))
    if model is not None:
        xx, yy = np.meshgrid(np.linspace(-1.7, 2.7, 300), np.linspace(-1.2, 1.7, 200))
        regions = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, regions, levels=[-0.5, 0.5, 1.5], colors=["#dceafe", "#fbe7d3"])
    for label, colour in [(0, "tab:blue"), (1, "tab:orange")]:
        ax.scatter(*X_train[y_train == label].T, s=30, color=colour, label=f"train {names[label]}")
        ax.scatter(*X_test[y_test == label].T, s=40, facecolors="none", edgecolors=colour,
                   label=f"test {names[label]}")
    ax.scatter(*unknown.T, s=150, marker="D", facecolors="white", edgecolors="black",
               linewidths=2, label="unknown")
    ax.set(xlim=(-1.7, 2.7), ylim=(-1.2, 1.7), xlabel="Feature 1", ylabel="Feature 2", title=title)
    ax.set_aspect("equal")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
    plt.show()

plot_model(title="Training data, test data and the unknown point")

**Predict before you run the next cell.** Which class would you give the diamond? Which points did you look at to decide?

## 5. k-nearest neighbours

KNN predicts the class of a new point from the classes of the `k` training examples closest to it. `fit` stores the training examples. `predict` gives a class for new points. `score` returns the accuracy, which is the proportion of examples it gets right.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

print("Prediction for the unknown point:", names[model.predict(unknown)[0]])
print("Training accuracy:", round(model.score(X_train, y_train), 3))
print("Test accuracy:", round(model.score(X_test, y_test), 3))
plot_model(model, "KNN with k = 3")

### Does `k` matter?

**Predict before you run the next cell.** Will the diamond keep the same class when `k` changes?

The loop below fits one model for each value of `k`, and collects the results in a `pandas` table.

In [ ]:
rows = []
for k in [1, 3, 9, 15]:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    rows.append({
        "model": f"KNN, k={k}",
        "train accuracy": model.score(X_train, y_train),
        "test accuracy": model.score(X_test, y_test),
        "unknown point": names[model.predict(unknown)[0]],
    })

knn_results = pd.DataFrame(rows)
knn_results.round(3)

With `k=1`, training accuracy is perfect, because every training example is its own nearest neighbour. Test accuracy is the lowest of the four. The diamond changes class somewhere between `k=3` and `k=9`.

### Try it yourself

1. Plot the regions for `k=1` and for `k=15` with `plot_model`. Where does the boundary change most?
2. Add other values of `k` to the loop, such as 5, 25 and 51. What happens to training and test accuracy as `k` grows?

In [ ]:
# Your experiments here

## 6. Decision trees

A decision tree learns a series of yes/no questions about the features. `max_depth` limits how many questions it can ask, one after another, before it gives an answer.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)

print("Prediction for the unknown point:", names[tree.predict(unknown)[0]])
print("Training accuracy:", round(tree.score(X_train, y_train), 3))
print("Test accuracy:", round(tree.score(X_test, y_test), 3))
plot_model(tree, "Decision tree with max_depth = 2")

Unlike KNN, a tree keeps a set of rules rather than the training examples. `export_text` prints the rules this tree learned. `class: 0` means class A, and `class: 1` means class B.

In [ ]:
print(export_text(tree, feature_names=["feature 1", "feature 2"]))

### What if the tree keeps growing?

**Predict before you run the next cell.** As the tree is allowed to grow deeper, will test accuracy rise along with training accuracy?

`max_depth=None` removes the limit completely.

In [ ]:
rows = []
for depth in [2, 5, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    rows.append({
        "model": f"Tree, depth={depth}",
        "nodes": tree.tree_.node_count,
        "leaves": tree.get_n_leaves(),
        "train accuracy": tree.score(X_train, y_train),
        "test accuracy": tree.score(X_test, y_test),
        "unknown point": names[tree.predict(unknown)[0]],
    })

tree_results = pd.DataFrame(rows)
tree_results.round(3)

In [ ]:
plot_model(tree, "Decision tree with no depth limit")

The unrestricted tree gets every training example right, but it does worst on the test set. It has learned rules for single training points that do not carry over to new data. This is **overfitting**.

### Try it yourself

1. Try `max_depth` values of 3 and 4. Which depth gives the best test accuracy?
2. Print the rules of the unrestricted tree with `export_text`. How many rules does it need?

In [ ]:
# Your experiments here

## 7. Compare all the models

`pd.concat` joins the two tables. `pandas` can also draw a quick bar chart straight from a table.

In [ ]:
all_results = pd.concat([knn_results, tree_results], ignore_index=True)
ax = all_results.plot.barh(x="model", y=["train accuracy", "test accuracy"], xlim=(0.6, 1.02),
                           figsize=(8, 5), title="Good on what data?")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()
all_results[["model", "train accuracy", "test accuracy"]].round(3)

The setting with the best training score is not the setting with the best test score. Training accuracy alone cannot tell you which model to trust.

## 8. Where accuracy goes blind

Accuracy is not always a good measure. Suppose 990 of 1,000 cases are normal and only 10 are the rare event we care about. A model that always predicts "normal" looks very accurate.

`DummyClassifier` is a baseline that ignores the features. With `strategy="most_frequent"` it always predicts the most common class.

In [ ]:
from sklearn.dummy import DummyClassifier

y_rare = np.array([0] * 990 + [1] * 10)
X_rare = np.zeros((1000, 1))

baseline = DummyClassifier(strategy="most_frequent").fit(X_rare, y_rare)
predictions = baseline.predict(X_rare)

print("Accuracy:", baseline.score(X_rare, y_rare))
print("Rare cases found:", np.sum((predictions == 1) & (y_rare == 1)), "of", np.sum(y_rare == 1))

The baseline scores 99% accuracy and finds none of the rare cases. The measure has to fit the problem.

## 9. What you practised

- Generating a dataset with a fixed `random_state`, so results can be repeated.
- Splitting data into training and test sets with `train_test_split`, and keeping the class mix with `stratify`.
- Fitting several models in a loop and collecting the results in a `pandas` table.
- Reading training and test accuracy side by side, and spotting overfitting.
- Printing the rules a decision tree learned with `export_text`.
- Using a simple baseline to check whether an accuracy figure means anything.